#### Root Path & Connection

In [2]:
import os
from dotenv import load_dotenv
import sqlalchemy as sa
from sqlalchemy import text
import pandas as pd
from datetime import datetime

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
    print("Changed to project root folder")

print("Current working directory:", os.getcwd())

load_dotenv(override=True)

# Create connection string
password = os.getenv('DB_PASSWORD')
connection_string = f"mysql+pymysql://{os.getenv('DB_USER')}:{password}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}?charset=utf8mb4"

engine = sa.create_engine(connection_string, pool_pre_ping=True, connect_args={"connect_timeout": 10})

print("Database engine created")

Changed to project root folder
Current working directory: C:\Users\tohiba\Desktop\eudr-coffee-traceability-romina
Database engine created


#### Load the improved dataset

In [5]:
raw_data_path = os.path.join('notebooks', 'data', 'processed', 'romina_compliance_final.csv')

df = pd.read_csv(raw_data_path)
print(f"Loaded {len(df)} farm plots for migration.")

df.columns = [c.lower().strip().replace(' ', '_') for c in df.columns]

date_cols = ['harvest_start_date', 'harvest_end_date', 'inspection_date', 'sync_date']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.date
    else:
        # If sync_date is missing, we create it now
        if col == 'sync_date':
            df['sync_date'] = datetime.now().date()

numeric_cols = ['latitude', 'longitude', 'eudr_readiness_score', 'plot_size_ha', 'altitude_masl']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates(subset=['farmer_id'])

print("Data transformation complete. Ready for 'to_sql' upload.")

Loaded 2000 farm plots for migration.
Data transformation complete. Ready for 'to_sql' upload.


#### Insert Data

In [11]:
def assign_risk(score):
    if score >= 80: return 'Low'
    if score >= 50: return 'Medium'
    return 'High'

df['risk_category'] = df['eudr_readiness_score'].apply(assign_risk)

sql_expected_columns = [
    'farmer_id', 'region', 'woreda', 'latitude', 'longitude', 
    'altitude_masl', 'plot_size_ha', 'coffee_variety', 
    'eudr_readiness_score', 'is_non_compliant', 'risk_category', 'sync_date'
]

df_final_sync = df[sql_expected_columns].copy()

try:
    df_final_sync.to_sql(
        'eudr_compliance_audit', 
        con=engine, 
        if_exists='append', 
        index=False, 
        method='multi', 
        chunksize=500
    )
    
    print(f"SUCCESS: {len(df_final_sync)} records migrated to 'eudr_compliance_audit'!")
    
    # Verify the Count
    with engine.connect() as conn:
        result = conn.execute(sa.text("SELECT COUNT(*) FROM eudr_compliance_audit"))
        print(f"Total records in MySQL: {result.scalar()}")

except Exception as e:
    print(f"Final Migration Error: {e}")
    print("If you see 'Duplicate entry', run 'TRUNCATE TABLE eudr_compliance_audit;' in Workbench.")

SUCCESS: 2000 records migrated to 'eudr_compliance_audit'!
Total records in MySQL: 2000
